In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/prasadshet/indian-sign-language-video-dataset/Video_Dataset/Video_Dataset/Fedup/WIN_20231103_13_04_01_Pro_right_tilt.mp4
/kaggle/input/datasets/prasadshet/indian-sign-language-video-dataset/Video_Dataset/Video_Dataset/Fedup/WIN_20231103_13_07_22_Pro_right_tilt.mp4
/kaggle/input/datasets/prasadshet/indian-sign-language-video-dataset/Video_Dataset/Video_Dataset/Fedup/WIN_20231103_13_07_22_Pro_left_tilt.mp4
/kaggle/input/datasets/prasadshet/indian-sign-language-video-dataset/Video_Dataset/Video_Dataset/Fedup/WIN_20231103_13_03_36_Pro_right_tilt.mp4
/kaggle/input/datasets/prasadshet/indian-sign-language-video-dataset/Video_Dataset/Video_Dataset/Fedup/WIN_20231103_13_06_17_Pro.mp4
/kaggle/input/datasets/prasadshet/indian-sign-language-video-dataset/Video_Dataset/Video_Dataset/Fedup/WIN_20231103_13_06_17_Pro_right_tilt.mp4
/kaggle/input/datasets/prasadshet/indian-sign-language-video-dataset/Video_Dataset/Video_Dataset/Fedup/WIN_20231103_13_02_30_Pro_right_tilt.mp4
/kag

In [2]:
!pip install mediapipe opencv-python-headless -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [3]:
import mediapipe as mp
import cv2
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

2026-05-03 14:22:24.732046: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777818145.140637      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777818145.255083      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777818146.268884      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777818146.268925      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777818146.268927      23 computation_placer.cc:177] computation placer alr

In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".task"):
            print(os.path.join(root, f))

/kaggle/input/datasets/riya12chauhan/hand-landmarker/hand_landmarker.task


In [5]:


MODEL_PATH = "/kaggle/input/datasets/riya12chauhan/hand-landmarker/hand_landmarker.task"

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2
)

_landmarker = vision.HandLandmarker.create_from_options(options)

print("✅ Landmarker initialized")

✅ Landmarker initialized


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1777818180.703085      85 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777818180.767996      87 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
import os, json, random

import cv2
import tensorflow as tf
from tensorflow import keras
from collections import Counter
# Feature config
POS_DIM = 126
VEL_DIM = 126   # ✅ FIXED
META_DIM = 6
SEQ_LEN = 30

FEATURE_DIM = 258   # ✅ FIXED

DATASET_DIR = "/kaggle/input/datasets/prasadshet/indian-sign-language-video-dataset/Video_Dataset/Video_Dataset"
OUTPUT_DIR = "/kaggle/working"


In [7]:
def _extract_frame(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = _landmarker.detect(mp_image)

    lm   = np.zeros(POS_DIM, dtype=np.float32)
    meta = np.zeros(META_DIM, dtype=np.float32)

    if result.hand_landmarks:
        for i, hand in enumerate(result.hand_landmarks[:2]):
            base = i * 63
            for j, pt in enumerate(hand):
                lm[base + j*3: base + j*3+3] = [pt.x, pt.y, pt.z]

            meta[i*3] = 1.0

    # normalize
    out = lm.reshape(2, 21, 3)
    for h in range(2):
        if np.all(out[h] == 0):
            continue
        out[h] -= out[h][0]
        s = np.max(np.abs(out[h]))
        if s > 1e-6:
            out[h] /= s

    return out.flatten(), meta

In [8]:
def video_to_sequence(path):
    cap = cv2.VideoCapture(path)
    frames_lm, frames_meta = [], []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        lm, meta = _extract_frame(frame)
        frames_lm.append(lm)
        frames_meta.append(meta)

    cap.release()

    if len(frames_lm) < 2:
        return None

    lm_arr = np.array(frames_lm)
    meta_arr = np.array(frames_meta)

    vel_arr = np.zeros((len(lm_arr), VEL_DIM))
    for t in range(1, len(lm_arr)):
        vel_arr[t] = lm_arr[t] - lm_arr[t-1]

    seq = np.concatenate([lm_arr, vel_arr, meta_arr], axis=1)

    idx = np.linspace(0, len(seq)-1, SEQ_LEN).astype(int)
    return seq[idx]

In [9]:
SELECTED_CLASSES = [
    "Hello",
    "Wrong",
    "Give",
    "Good Morning",
    "Clean",
    "Thank you",
    "Still"
]

def load_dataset():
    X, y = [], []
    classes = SELECTED_CLASSES

    print("Selected Classes:", classes)

    for ci, cls in enumerate(classes):
        cls_path = os.path.join(DATASET_DIR, cls)

        if not os.path.exists(cls_path):
            print(f"⚠️ Missing class: {cls}")
            continue

        files = os.listdir(cls_path)
        print(f"{cls}: {len(files)}")

        for f in files:
            seq = video_to_sequence(os.path.join(cls_path, f))

            if seq is not None:
                X.append(seq)
                y.append(ci)

    return np.array(X), np.array(y), classes

In [10]:
X, y, classes = load_dataset()

print("Dataset shape:", X.shape)
print("Classes:", classes)

Selected Classes: ['Hello', 'Wrong', 'Give', 'Good Morning', 'Clean', 'Thank you', 'Still']
Hello: 60


W0000 00:00:1777818181.096628      84 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Wrong: 60
Give: 60
Good Morning: 60
Clean: 60
Thank you: 60
Still: 30
Dataset shape: (390, 30, 258)
Classes: ['Hello', 'Wrong', 'Give', 'Good Morning', 'Clean', 'Thank you', 'Still']


In [11]:
def build_model(num_classes):
    model = keras.Sequential([
        keras.layers.Input(shape=(SEQ_LEN, FEATURE_DIM)),

        keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True)),
        keras.layers.Dropout(0.5),

        keras.layers.Bidirectional(keras.layers.LSTM(32)),
        keras.layers.Dropout(0.5),

        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.4),

        keras.layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [12]:
model = build_model(len(classes))
model.summary()

I0000 00:00:1777820533.778739      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777820533.784644      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 30, 128)        │       165,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 211,207 (825.03 KB)

 Trainable params: 211,207 (825.03 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
idx = np.random.permutation(len(X))
X = X[idx]
y = y[idx]

In [14]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    )
]

In [15]:
model = build_model(len(classes))

history = model.fit(
    X, y,
    validation_split=0.2,
    epochs=20,
    batch_size=16,
    callbacks=callbacks
)

Epoch 1/20


I0000 00:00:1777820540.086818    2833 cuda_dnn.cc:529] Loaded cuDNN version 91002


20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.1441 - loss: 1.9496 - val_accuracy: 0.2949 - val_loss: 1.8803
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2508 - loss: 1.8964 - val_accuracy: 0.3333 - val_loss: 1.8070
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2759 - loss: 1.8000 - val_accuracy: 0.6026 - val_loss: 1.6670
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3702 - loss: 1.6496 - val_accuracy: 0.7436 - val_loss: 1.3888
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5082 - loss: 1.4349 - val_accuracy: 0.8590 - val_loss: 1.0355
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6254 - loss: 1.1110 - val_accuracy: 0.8846 - val_loss: 0.7065
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7601 - loss: 0.8815 - val_accuracy: 0.9359 - val_loss: 0.4705
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8503 - loss: 0.6480 - val_accuracy: 0.9359 - val_loss: 0.

In [16]:
model.save("/kaggle/working/isl_model.h5")

import json
with open("/kaggle/working/labels.json", "w") as f:
    json.dump(classes, f)

print("✅ Model saved")

✅ Model saved
